In [1]:
import sys
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add dirctory with model moduels to path
sys.path.insert(0, os.path.join(os.getcwd(),'..'))

import CIBUSmod as cm
from CIBUSmod.utils.output_data_manip import concat_herds

# Create input data
x0_crp.csv and x0_ani.csv represent the current areas of different crops and number of heads of different animals respectively. These are both still WIP, which is why some mangling is needed here to get them in the right form.
The demand vector is generated with demand for cattle meat, pig meat and cattle milk equal to what is in the agricultural statistics.

In [2]:
# Define x0_crp
x0_crp = \
    pd.read_csv(os.path.join('..','data','x0','x0_crp.csv'), dtype={'region': object})\
    .set_index(['crop','prod_system','region'])['area']

# Define x0_ani
x0_ani = pd.read_csv(os.path.join('..','data','x0','x0_ani.csv'), dtype={'region': object})

x0_ani['species'] = np.nan
x0_ani['species'] = \
np.where(np.isin(x0_ani['animal'], ['kor för mjölkproduktion', 'kor för uppfödning av kalvar']),'cattle',x0_ani['species'])
x0_ani['species'] = \
np.where(x0_ani['animal']=='suggor för avel','pigs',x0_ani['species'])

x0_ani['breed'] = np.nan
x0_ani['breed'] = \
np.where(x0_ani['animal']=='kor för mjölkproduktion','dairy',x0_ani['breed'])
x0_ani['breed'] = \
np.where(x0_ani['animal']=='kor för uppfödning av kalvar','beef',x0_ani['breed'])
x0_ani['breed'] = \
np.where(x0_ani['animal']=='suggor för avel','none',x0_ani['breed'])

x0_ani = x0_ani[x0_ani['species']!='nan'][['species','breed','prod_system','region','number']].set_index(['species','breed','prod_system','region'])['number']

x0_ani = x0_ani.fillna(0)

x0 = {'ani':x0_ani,'crp':x0_crp}

In [3]:
# Create demand vectors
# Animal products
D_ani = pd.Series(
    {
        ('conventional','cattle','meat') : (136-21) * 1000000,
        ('organic','cattle','meat')      : 21 * 1000000,
        ('conventional','cattle','milk') : (2760-464) * 1000000,
        ('organic','cattle','milk')      : 464 * 1000000,
        ('conventional','pigs','meat')   : (249-6) * 1000000,
        ('organic','pigs','meat')        : 6 * 1000000,
    },
)
D_ani.index.rename(['prod_system','species','product'], inplace=True)

# Crop products (kg DM)
D_crp = pd.Series(
    {
        ('conventional','wheat')         :0,
        ('organic','wheat')              :0,
    },
)
D_crp.index.rename(['prod_system','crop_product'], inplace=True)

D = {'ani':D_ani, 'crp':D_crp}

## Work on Diet class

In [80]:
# Instantiate manure management
diet = cm.Diet(
    par = cm.ParameterRetriever(
        os.path.join('..','data','prod_parameters','Diet.xlsx')
    )
)

# Calculate food demand
diet.calculate(verbose=True)

[11:18:34][Diet] Calculating food demand ...
[11:18:34][Diet] Calculating waste ...
[11:18:34][Diet] Calculating crop product demand ...
[11:18:34][Diet] Calculating crop product demand ...
[11:18:34][Diet] Done! Elapsed time: 0 sec


In [81]:
round(diet.food_demand/1000,0)

origin                                           domestic  imported
food                               prod_system                     
Wheat and products                 conventional  261533.0   73766.0
                                   organic        13765.0    3882.0
Rice and products                  conventional       0.0   53454.0
                                   organic            0.0       0.0
Barley and products                conventional   11464.0     478.0
                                   organic         1274.0      53.0
Milk and products less than 1% fat conventional  114220.0    6445.0
                                   organic        23394.0       0.0
Milk and products 1-2% fat         conventional  294753.0   11090.0
                                   organic        60371.0       0.0
Milk and products more than 2% fat conventional  324004.0   42570.0
                                   organic        66362.0       0.0
Cheese                             conventional   58904.0  106452.0
                                   organic        12065.0       0.0
Icecream                           conventional   25487.0       0.0
                                   organic         5220.0       0.0
Sourcream                          conventional    2517.0     758.0
                                   organic          516.0       0.0
Cream low fat                      conventional   16614.0    5004.0
                                   organic         3403.0       0.0
Cream high fat                     conventional   20893.0    6293.0
                                   organic         4279.0       0.0
Butter                             conventional   27441.0   24941.0
                                   organic         5620.0       0.0
Pig meat and products              conventional  144091.0   82705.0
                                   organic         2941.0       0.0

In [82]:
round(diet.food_demand_to_processing/1000,2)

origin                                            domestic   imported
food                               prod_system                       
Wheat and products                 conventional  397571.82  112135.64
                                   organic        20924.83    5901.88
Rice and products                  conventional       0.00   81258.13
                                   organic            0.00       0.00
Barley and products                conventional   17427.28     726.14
                                   organic         1936.36      80.68
Milk and products less than 1% fat conventional  130285.03    7351.21
                                   organic        26684.89       0.00
Milk and products 1-2% fat         conventional  336209.22   12650.24
                                   organic        68862.13       0.00
Milk and products more than 2% fat conventional  369575.09   48557.02
                                   organic        75696.10       0.00
Cheese                             conventional   67188.31  121424.66
                                   organic        13761.46       0.00
Icecream                           conventional   29071.87       0.00
                                   organic         5954.48       0.00
Sourcream                          conventional    2871.30     864.85
                                   organic          588.10       0.00
Cream low fat                      conventional   18950.55    5708.00
                                   organic         3881.44       0.00
Cream high fat                     conventional   23831.75    7178.24
                                   organic         4881.20       0.00
Butter                             conventional   31300.71   28449.18
                                   organic         6410.99       0.00
Pig meat and products              conventional  177521.89  101893.94
                                   organic         3622.90       0.00

In [83]:
round(diet.waste/(diet.population*1000000),2)

waste_level                                      household  retail  processing
food                               prod_system                                
Wheat and products                 conventional      10.77    0.88        4.02
                                   organic            0.57    0.05        0.21
Rice and products                  conventional       1.72    0.14        0.00
                                   organic            0.00    0.00        0.00
Barley and products                conventional       0.38    0.03        0.18
                                   organic            0.04    0.00        0.02
Milk and products less than 1% fat conventional       1.44    0.07        0.13
                                   organic            0.28    0.01        0.03
Milk and products 1-2% fat         conventional       3.64    0.17        0.32
                                   organic            0.72    0.03        0.07
Milk and products more than 2% fat conventional       4.37    0.20        0.36
                                   organic            0.79    0.04        0.07
Cheese                             conventional       1.97    0.09        0.06
                                   organic            0.14    0.01        0.01
Icecream                           conventional       0.30    0.01        0.03
                                   organic            0.06    0.00        0.01
Sourcream                          conventional       0.04    0.00        0.00
                                   organic            0.01    0.00        0.00
Cream low fat                      conventional       0.26    0.01        0.02
                                   organic            0.04    0.00        0.00
Cream high fat                     conventional       0.32    0.01        0.02
                                   organic            0.05    0.00        0.00
Butter                             conventional       0.62    0.03        0.03
                                   organic            0.07    0.00        0.01
Pig meat and products              conventional       2.70    1.02        0.86
                                   organic            0.04    0.01        0.02

In [84]:
round(diet.crop_prod_demand/(diet.population*1000000),2)

prod_system   crop_prod
conventional  wheat        48.49
              barley        1.85
organic       wheat         2.55
              barley        0.21
dtype: float64

In [85]:
round(diet.crop_by_products/(diet.population*1000000),2)

prod_system   crop_prod  by_prod
conventional  barley     bran       0.00
                         hulls      0.15
              wheat      bran       7.28
                         germ       0.77
organic       barley     bran       0.00
                         hulls      0.02
              wheat      bran       0.38
                         germ       0.04
dtype: float64

In [86]:
round(diet.animal_prod_demand/(diet.population*1000000),2)

prod_system   species  animal_prod
conventional  cattle   milk           209.41
              pigs     meat            23.43
organic       cattle   milk            42.89
              pigs     meat             0.48
dtype: float64

In [87]:
round(diet.animal_prod_demand/(diet.population*1000000),2)

prod_system   species  animal_prod
conventional  cattle   milk           209.41
              pigs     meat            23.43
organic       cattle   milk            42.89
              pigs     meat             0.48
dtype: float64

In [88]:
round(diet.animal_by_products/(diet.population*1000000),2)

prod_system   species  animal_prod  by_prod    
conventional  cattle   milk         buttermilk     1.53
                                    whey           5.43
              pigs     meat         butcher fat    3.08
                                    fat            1.35
                                    offals         0.90
                                    skins          1.13
organic       cattle   milk         buttermilk     0.31
                                    whey           1.11
              pigs     meat         butcher fat    0.06
                                    fat            0.03
                                    offals         0.02
                                    skins          0.02
dtype: float64

In [89]:
round(diet.induced_skim_milk_exports/(diet.population*1000000),2)

prod_system
conventional    49.64
organic         10.17
dtype: float64

In [ ]:
######################################################
######################################################

# Run base year
This runs the model for the base year (2016-2020). Results are then compared to statistics to ensure feasible results replicating the current situation.

In [ ]:
# Instantiate crop production
crops = cm.CropProduction(
    par = cm.ParameterRetriever(
        os.path.join('..','data','prod_parameters','CropProduction.xlsx')
    ),
    index = x0_crp.index
)    

# Instantiate animal herds
herds=pd.Series(
    data=[],
    index=pd.MultiIndex(
        levels=[[]]*4,
        codes=[[]]*4,
        names=['species','breed','prod_system','sub_system']
    ),
    dtype = object
)

for (sp,br,ps) in x0_ani.groupby(['species','breed','prod_system']).sum().index:
            
    if sp == 'cattle':
        herds[(sp,br,ps,'none')] = \
            cm.CattleHerd(
                par = cm.ParameterRetriever(
                    os.path.join('..','data','prod_parameters','CattleHerd.xlsx')
                ),
                index = x0_ani.index.get_level_values('region').unique(),
                breed = br,
                prod_system = ps
            )

    elif sp == 'pigs':
        herds[(sp,br,ps,'none')] = \
            cm.PigHerd(
                par = cm.ParameterRetriever(
                    os.path.join('..','data','prod_parameters','PigHerd.xlsx')
                ),
                index = x0_ani.index.get_level_values('region').unique(),
                breed = br,
                prod_system = ps
            ) 

# Instantiate manure management
feed_mgmt = cm.FeedMgmt(
    herds = herds,
    par = cm.ParameterRetriever(
        os.path.join('..','data','prod_parameters','FeedMgmt.xlsx')
    )
)

# Instantiate manure management
manure_mgmt = cm.ManureMgmt(
    herds = herds,
    par = cm.ParameterRetriever(
        os.path.join('..','data','prod_parameters','ManureMgmt.xlsx')
    )
)

# Instantiate geo distributor
geodist = cm.GeoDistributor(D,x0,crops,herds,feed_mgmt)

# -------------------------------------------------------------------- #

# Calculate crops
crops.calculate(
    verbose = True
)

# Calculate herds
for h in herds:
    h.calculate(verbose = True)

# Calculate feed
feed_mgmt.calculate(verbose=True)    

# Calculate manure
manure_mgmt.calculate(verbose=True)

In [ ]:
# Distribute animals and crops
geodist.make(use_cons=[1,2,3,4],verbose=True)


In [ ]:
geodist.solve(verbose=True)

In [ ]:
# Scale and store results
out_crops = crops.scale(geodist.x['crp'])

out_animals = concat_herds([
    h.scale(
        geodist.x['ani'].loc[(h.species,h.breed,h.prod_system,h.sub_system)],
        x_is = h.x_is
    )
    for h in herds
])

In [ ]:
fig, ax = plt.subplots(figsize=(15,5))

pd.concat([
    x0_crp.groupby('crop').sum().rename('x0'),
    out_crops.area.groupby('crop').sum().rename('x')
], axis=1).plot.bar(ax=ax)

plt.show()

In [ ]:
idx = pd.IndexSlice

print(
    pd.concat([
        out_animals.heads
        .groupby(['species','breed','prod_system','animal'], axis=1).sum()
        .sum().loc[idx[:,:,:,['cows','sows']]].droplevel('animal')
        .rename('x')
        ,
        x0_ani
        .groupby(['species','breed','prod_system']).sum()
        .rename('x0')
    ], axis=1)
    .apply(
        lambda x:
        pd.Series(
            [x['x'].round(1),x['x0'].round(1),((x['x']-x['x0'])/x['x0']*100).round(1)],
            index = ['x','x0','% dif']
        ),
        axis=1
    )
)

print(
    pd.concat([
        out_crops.area.groupby('crop').sum().rename('x'),
        x0_crp.groupby('crop').sum().rename('x0')
    ], axis=1)
    .apply(
        lambda x:
        pd.Series(
            [x['x'].round(1),x['x0'].round(1),((x['x']-x['x0'])/x['x0']*100).round(1)],
            index = ['x','x0','% dif']
        ),
        axis=1
    )
)